# Northstar Code Arena — Colab coursework

Sign in to your own Google account, choose the Colab runtime you need, then run the single code cell below once. Paste the connection code copied by Code Arena when prompted.

The connection code is single-use and short-lived. A result from this notebook is labelled **Colab validation**; it is not an external Online Judge verdict.

In [ ]:
# Run this one cell. It is safe to rerun after a result-delivery timeout.
from getpass import getpass
import base64
import contextlib
import hashlib
import importlib.util
from importlib.metadata import PackageNotFoundError, version as package_version
import io
import json
from packaging.version import Version
import re
import requests
import shutil
import subprocess
import time
import traceback
from urllib.parse import urlsplit
import uuid
from IPython.display import Code, display

def northstar_api_data(response):
    try:
        payload = response.json()
    except ValueError as error:
        raise RuntimeError(f"Platform returned a non-JSON response ({response.status_code})") from error
    if not response.ok or payload.get("ok") is not True:
        message = payload.get("error", {}).get("message", response.text)
        raise RuntimeError(f"Platform request failed ({response.status_code}): {message}")
    return payload["data"]

def decode_connection_code(value):
    value = value.strip()
    if not value.startswith("NS1."):
        raise RuntimeError("This is not a Northstar Colab connection code.")
    encoded = value[4:]
    if not re.fullmatch(r"[A-Za-z0-9_-]+", encoded):
        raise RuntimeError("The Colab connection code is malformed.")
    try:
        padding = "=" * ((4 - len(encoded) % 4) % 4)
        details = json.loads(base64.urlsafe_b64decode(encoded + padding).decode("utf-8"))
    except (ValueError, UnicodeDecodeError, json.JSONDecodeError) as error:
        raise RuntimeError("The Colab connection code could not be decoded.") from error
    api_base = details.get("apiBase") if isinstance(details, dict) else None
    pairing_code = details.get("pairingCode") if isinstance(details, dict) else None
    if not isinstance(api_base, str) or not isinstance(pairing_code, str):
        raise RuntimeError("The Colab connection code is incomplete.")
    parsed_api = urlsplit(api_base)
    if (parsed_api.scheme != "https" or not parsed_api.hostname or parsed_api.username or
            parsed_api.password or parsed_api.query or parsed_api.fragment or parsed_api.path not in ("", "/")):
        raise RuntimeError("The connection code does not contain a valid public HTTPS API origin.")
    if not re.fullmatch(r"[A-Z0-9]{4}-[A-Z0-9]{4}", pairing_code):
        raise RuntimeError("The connection code contains an invalid pairing secret.")
    return api_base.rstrip("/"), pairing_code

def json_safe(value):
    try:
        return json.loads(json.dumps(value))
    except (TypeError, ValueError):
        return {"repr": repr(value)}

def colab_environment_metrics():
    if not shutil.which("nvidia-smi"):
        return {"colabGpuAvailable": False}
    query = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
        capture_output=True, text=True, check=False,
    )
    line = query.stdout.strip().splitlines()[0] if query.stdout.strip() else ""
    if query.returncode != 0 or not line:
        return {"colabGpuAvailable": False}
    name, memory_mib = [part.strip() for part in line.split(",", 1)]
    return {"colabGpuAvailable": True, "colabGpuName": name, "gpuMemoryMiB": int(memory_mib)}

CONNECTION_CODE = getpass("Paste the connection code copied by Code Arena: ").strip()
connection_digest = hashlib.sha256(CONNECTION_CODE.encode("utf-8")).hexdigest()
connection_cache = globals().get("__northstar_connection_cache__")

if isinstance(connection_cache, dict) and connection_cache.get("digest") == connection_digest:
    API_BASE = connection_cache["apiBase"]
    RUN_ID = connection_cache["runId"]
    AUTH = connection_cache["auth"]
    bundle = connection_cache["bundle"]
    print("Reusing the paired session for an idempotent retry.")
else:
    API_BASE, PAIRING_CODE = decode_connection_code(CONNECTION_CODE)
    claim = northstar_api_data(requests.post(
        f"{API_BASE}/api/colab/claim",
        json={"pairingCode": PAIRING_CODE},
        timeout=30,
    ))
    RUN_ID = claim["runId"]
    RUN_TOKEN = claim["accessToken"]
    AUTH = {"Authorization": f"Bearer {RUN_TOKEN}"}
    del PAIRING_CODE, RUN_TOKEN
    bundle = northstar_api_data(requests.get(
        f"{API_BASE}/api/colab/runtime/{RUN_ID}/bundle",
        headers=AUTH,
        timeout=30,
    ))
    __northstar_connection_cache__ = {
        "digest": connection_digest,
        "apiBase": API_BASE,
        "runId": RUN_ID,
        "auth": AUTH,
        "bundle": bundle,
    }
del CONNECTION_CODE

execution = bundle["execution"]
runtime = execution["runtime"]
missing_dependencies = []
outdated_dependencies = []
for dependency in runtime["dependencies"]:
    import_root = dependency["importName"].split(".", 1)[0]
    if importlib.util.find_spec(import_root) is None:
        missing_dependencies.append(dependency["package"])
        continue
    minimum = dependency.get("minimumVersion")
    if minimum:
        try:
            installed = package_version(dependency["package"])
            if Version(installed) < Version(minimum):
                outdated_dependencies.append(f"{dependency['package']} {installed} (needs >= {minimum})")
        except PackageNotFoundError:
            missing_dependencies.append(dependency["package"])
if missing_dependencies or outdated_dependencies:
    problems = [*(f"missing {name}" for name in missing_dependencies), *outdated_dependencies]
    raise RuntimeError("Colab runtime requirements are not satisfied: " + "; ".join(problems))

print(f"Connected to activity: {bundle['activityId']}")
print(f"Entry point: {execution['entryPoint']}")
print(f"Trusted validation harness: {execution['validationHarnessRef']}")
print(f"Requested accelerator: {runtime['accelerator']}")
print(f"Validation cases loaded: {len(bundle['validation']['tests'])}")

callback_cache = globals().get("__northstar_callback_cache__")
if isinstance(callback_cache, dict) and callback_cache.get("runId") == RUN_ID:
    callback = callback_cache["callback"]
    print("Retrying the previously prepared result with the same execution ID.")
else:
    print("\nCode received from Code Arena:")
    display(Code(bundle["sourceCode"], language="python"))
    environment_metrics = colab_environment_metrics()
    accelerator = runtime["accelerator"]
    minimum_gpu_memory = runtime.get("minimumGpuMemoryMb")
    runtime_requirement_error = None
    if accelerator == "gpu_required" and not environment_metrics.get("colabGpuAvailable"):
        runtime_requirement_error = "This activity requires an NVIDIA GPU. In Colab, choose Runtime > Change runtime type, select a GPU, and reconnect."
    elif minimum_gpu_memory and environment_metrics.get("gpuMemoryMiB", 0) < minimum_gpu_memory:
        runtime_requirement_error = f"This activity requires at least {minimum_gpu_memory} MiB of GPU memory."

    captured_stdout = io.StringIO()
    captured_stderr = io.StringIO()
    namespace = {
        "__name__": "__main__",
        "__northstar_tests__": bundle["validation"]["tests"],
    }
    execution_error = runtime_requirement_error
    execution_started_at = time.perf_counter()
    with contextlib.redirect_stdout(captured_stdout), contextlib.redirect_stderr(captured_stderr):
        if execution_error is None:
            try:
                exec(compile(bundle["sourceCode"], "student_submission.py", "exec"), namespace)
                exec(compile(bundle["validation"]["harnessCode"], "northstar_validation.py", "exec"), namespace)
            except (Exception, SystemExit):
                execution_error = traceback.format_exc()
    execution_seconds = time.perf_counter() - execution_started_at
    if execution_error is None and execution_seconds > runtime["maxExecutionSeconds"]:
        execution_error = f"Execution exceeded the {runtime['maxExecutionSeconds']} second activity limit."

    validation = namespace.get("__northstar_validation__")
    if execution_error is not None:
        validation = {"status": "error", "tests": [], "metrics": {}}
    elif not isinstance(validation, dict):
        execution_error = "Validation harness did not produce __northstar_validation__."
        validation = {"status": "error", "tests": [], "metrics": {}}

    metrics = validation.get("metrics", {})
    if not isinstance(metrics, dict):
        metrics = {"reportedMetrics": json_safe(metrics)}
    metrics.update(environment_metrics)
    metrics["executionSeconds"] = round(execution_seconds, 4)
    stdout = captured_stdout.getvalue()
    stderr = captured_stderr.getvalue() + (execution_error or "")
    output_truncated = len(stdout) > 50000 or len(stderr) > 50000
    callback = {
        "executionId": str(uuid.uuid4()),
        "status": validation.get("status", "error"),
        "tests": json_safe(validation.get("tests", [])),
        "returnValue": json_safe(validation.get("returnValue")),
        "metrics": json_safe(metrics),
        "stdout": stdout[:50000],
        "stderr": stderr[:50000],
        "outputTruncated": output_truncated,
    }
    __northstar_callback_cache__ = {"runId": RUN_ID, "callback": callback}

print("\nColab validation evidence:")
print(json.dumps(callback, indent=2))
accepted = northstar_api_data(requests.post(
    f"{API_BASE}/api/colab/runtime/{RUN_ID}/result",
    headers={**AUTH, "Content-Type": "application/json"},
    json=callback,
    timeout=30,
))
print(f"\nColab validation returned to Code Arena: {accepted['result']['status']}")
print("Return to Code Arena. If delivery timed out instead, rerun this same cell with the same connection code.")

After the result is returned, go back to Code Arena. To try changed code, edit it there and choose **Retry latest code in Colab**; this creates a fresh immutable validation run.